In [ ]:
# ======================================================
# 0. 코랩에서 구글 드라이브 마운트하기
# ======================================================
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/kaggle-learn/secom'

import os
print(os.listdir(DATA_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['secom.data', 'secom_labels.data', 'secom.names', 'secom_report.md']


In [ ]:
# ======================================================
# 1. 데이터 읽기 - [L1] read_csv로 센서 데이터와 라벨 파일 읽기
# ======================================================
import pandas as pd

X = pd.read_csv(f'{DATA_DIR}/secom.data', sep=r'\s+', header=None)
y = pd.read_csv(f'{DATA_DIR}/secom_labels.data', sep=r'\s+', header=None)

# y 데이터프레임을 불러온 직후에 컬럼 이름을 직접 지정해야 한다.
y.columns = ['label', 'timestamp']

print(X.shape) # 행 1567, 열 590개
print(y.shape) # (1567, 2)

y.head() # 라벨 -1 : 합격(정상) / 1 : 불합격(불량)

(1567, 590)
(1567, 2)


,label,timestamp
0,-1,19/07/2008 11:55:00
1,-1,19/07/2008 12:32:00
2,1,19/07/2008 13:17:00
3,-1,19/07/2008 14:43:00
4,-1,19/07/2008 15:22:00


In [ ]:
# ======================================================
# 2. 열 이름 붙이기 - [L6] 열 이름 붙이기
# ======================================================
X.columns = [f'sensor_{i:03d}' for i in range(X.shape[1])]
Y = y.rename(columns={0: 'label', 1:'timestamp'})

print(X.columns[:5].tolist())
y.head(3)

['sensor_000', 'sensor_001', 'sensor_002', 'sensor_003', 'sensor_004']


,label,timestamp
0,-1,19/07/2008 11:55:00
1,-1,19/07/2008 12:32:00
2,1,19/07/2008 13:17:00


**[i:03d]**
정수를 3자리로 표현하고, 빈 자리를 0으로 채우는 서식 지정자

**왜 X는 대문자로, y는 소문자로 쓰나요?**
머신러닝과 통계학 코드를 작성할 때 특징(Feature) 데이터를 X(대문자)로, 정답/레이블(Target/Label) 데이터를 y(소문자)로 쓰는 것은 전 세계 데이터 사이언스 업계의 오랜 표준 관례이다.

In [ ]:
# ======================================================
# 3. 센서와 라벨 합치기 - [L6] pd.concat(axis=1)
# ======================================================
df = pd.concat([y, X], axis=1).copy()
print(df.shape) # (1567, 592) = 라벨 2열 + 센서 590열

'''
[ 아래 코드가 안되는 이유 ]
전체 592개 열이 있는데 왜 이름은 2개만 주니? -> ValueError
합치기 전(각각 따로 있을 때)에 미리 이름을 정리한다.

'''
# df.columns = ['label', 'timestamp']

(1567, 592)


**pd.concat([y, X], axis=1)** 두 데이터프레임(y와 X)을 행(row)을 기준으로 좌우로(열방향으로) 합쳐서 하나의 커다란 데이터프레임으로 만드는 작업을 의미한다.

In [ ]:
# ======================================================
# 4. 자료형 확인과 변환 — [L5] dtypes, astype, pd.to_datetime
# ======================================================
print(df[['label', 'timestamp', 'sensor_000']].dtypes)
print(X.dtypes.value_counts()) # float64 590개

df['timestamp'] = pd.to_datetime(df['timestamp'], format='%d/%m/%Y %H:%M:%S')
df['label_cat'] = df['label'].astype('category')

print(df[['timestamp','label_cat']].dtypes)

label                  int64
timestamp     datetime64[ns]
sensor_000           float64
dtype: object
float64    590
Name: count, dtype: int64
timestamp    datetime64[ns]
label_cat          category
dtype: object


**format='%d/%m/%Y %H:%M:%S'**


*   %d: 일(Day) — 2자리 숫자로 표시 (예: 01, 15, 31)
*   %m: 월(Month) — 2자리 숫자로 표시 (예: 01, 05, 12)
*   %Y: 연도(Year) — 4자리 숫자로 표시 (예: 2023, 2026)
*   %H: 시간(Hour) — 24시간 형식의 시간 (00~23)
*   %M: 분(Minute) — 2자리 숫자로 표시 (00~59)
*   %S: 초(Second) — 2자리 숫자로 표시 (00~59)

In [ ]:
# ======================================================
# 5. 원하는 데이터 골라내기 - [L2] iloc, loc, 조건 선택, 새 열 대입
# ======================================================

# 위치 기준 : 앞 5행, 앞 4열
print(df.iloc[:5,:4])

# 이름 기준 : 열 범위 sensor_000 ~ sensor_002 (끝 포함)
print(df.loc[:4, 'sensor_000':'sensor_002'])

# 조건 선택 : 불합격 제품만
fail = df.loc[df['label'] == 1]
print('불합격 건수 : ', len(fail))


# 조건 두 개 묶기 : 불합격이면서 2008년 8월에 측정된 제품
fail_aug = df.loc[(df['label'] == 1) & (df['timestamp'].dt.month == 8)]
print('8월 불합격 : ', len(fail_aug))
print()
print()

# 새 열 대입 : 불합격이면 1, 합격이면 0(머신러닝에서 쓰기 쉬운 형태)
df['is_fail'] = (df['label'] == 1).astype(int)
print(df['is_fail'])
print()

df[['label','is_fail']].head()

   label           timestamp  sensor_000  sensor_001
0     -1 2008-07-19 11:55:00     3030.93     2564.00
1     -1 2008-07-19 12:32:00     3095.78     2465.14
2      1 2008-07-19 13:17:00     2932.61     2559.94
3     -1 2008-07-19 14:43:00     2988.72     2479.90
4     -1 2008-07-19 15:22:00     3032.24     2502.87
   sensor_000  sensor_001  sensor_002
0     3030.93     2564.00   2187.7333
1     3095.78     2465.14   2230.4222
2     2932.61     2559.94   2186.4111
3     2988.72     2479.90   2199.0333
4     3032.24     2502.87   2233.3667
불합격 건수 :  104
8월 불합격 :  51


0       0
1       0
2       1
3       0
4       0
       ..
1562    0
1563    0
1564    0
1565    0
1566    0
Name: is_fail, Length: 1567, dtype: int64



,label,is_fail
0,-1,0
1,-1,0
2,1,1
3,-1,0
4,-1,0


dt는 Datetime Accessor(날짜/시간 접근자)의 약자이다. 판다스(Pandas)에서 날짜아 시간 타입(datetime64)으로 변환된 열(Column) 뒤에 붙여서, 날짜/시간 관련 속성(연도,월,일,요일 등)이나 기능에 접근할 수 있게 해주는 특별한 도구함이다.

In [ ]:
# ======================================================
# 6. 요약하고 값 바꾸기 - [L3] describe, value_counts, map, apply
# ======================================================
print(df[['sensor_000', 'sensor_001', 'sensor_002']].describe())
print()
print()

print(df['label'].value_counts())
print()
print(df['label'].value_counts(normalize=True).round(3))
print()
print()

# map : -1/1 숫자를 사람이 읽기 쉬운 글자로
print("====== map ======")
df['result'] = df['label'].map({
    -1 : '합격',
    1 : '불합격'
})
print(df['result'].value_counts())
print()
print()


# apply(axis=0, 기본값) : 센서 열마다 '서로 다른 값의 개수'를 계산
print("====== apply ======")
sensor_cols = [c for c in df.columns if c.startswith('sensor_')]
n_unique = df[sensor_cols].apply(lambda col:col.nunique())
print("값이 1개 이하뿐인(변화 없는) 센서 수 : ", (n_unique <= 1).sum())

        sensor_000   sensor_001   sensor_002
count  1561.000000  1560.000000  1553.000000
mean   3014.452896  2495.850231  2200.547318
std      73.621787    80.407705    29.513152
min    2743.240000  2158.750000  2060.660000
25%    2966.260000  2452.247500  2181.044400
50%    3011.490000  2499.405000  2201.066700
75%    3056.650000  2538.822500  2218.055500
max    3356.350000  2846.440000  2315.266700


label
-1    1463
 1     104
Name: count, dtype: int64

label
-1    0.934
 1    0.066
Name: proportion, dtype: float64


====== map ======
result
합격     1463
불합격     104
Name: count, dtype: int64


====== apply ======
값이 1개 이하뿐인(변화 없는) 센서 수 :  116


value_counts(normalize=True) : 판다스(Pandas)에서 특정 열(Series)에 들어 있는 데이터의 종류별 개수(빈도)를 구한 뒤, 이를 전체 개수 대비 비율(백분율의 소수 형태)로 변환해주는  메서드이다.

map()은 판다스(Pandas)에서 시리즈(열) 안의 값들을 사용자가 원하는 다른 값으로 1:!로 짝지어(변환하여) 바꾸어 주는 메서드이다.

apply() : 각 값을 하나씩 꺼내서 뒤의 함수를 적용한다.
lambda col : 일회용 함수를 만드는 문법
nunique() : 고유한 값의 개수를 센다.

In [ ]:
# ======================================================
# 7. 묶어서 비교하고 정렬하기 - [L4] groupby, agg, sort_values, size
# ======================================================
# 합격/불합격 그룹별 센서 평균 비교
print(df.groupby('result')[['sensor_000','sensor_001','sensor_002']].agg(['mean','std']).round(2))
print()
print()



# 날짜별 생산 건수와 불합격 건수
daily = (df.groupby(df['timestamp'].dt.date)
          .agg(total=('is_fail', 'size'), fails=('is_fail', 'sum')))
daily['fail_rate'] = (daily['fails'] / daily['total']).round(3)
print()
print()


# 불합격이 많았던 날짜 상위 5일
print(daily.sort_values('fails', ascending=False).head())
print()
print()


# 합격/불합격 평균 차이가 가장 큰 센서 상위 10개(단위가 달라서 표준 편차로 나눠 비교)
means = df.groupby('result')[sensor_cols].mean()
gap = ((means.loc['불합격'] - means.loc['합격']).abs() / df[sensor_cols].std()).sort_values(ascending=False)
gap.head(10)


       sensor_000        sensor_001        sensor_002       
             mean    std       mean    std       mean    std
result                                                      
불합격       3007.53  88.41    2495.06  71.32    2200.44  31.30
합격        3014.95  72.46    2495.91  81.03    2200.55  29.39




            total  fails  fail_rate
timestamp                          
2008-10-05     32      6      0.188
2008-08-20     28      6      0.214
2008-08-17     17      5      0.294
2008-08-18     40      5      0.125
2008-08-10     12      5      0.417




,0
sensor_059,0.624373
sensor_103,0.606868
sensor_510,0.528162
sensor_348,0.519064
sensor_158,0.517005
sensor_111,0.500893
sensor_293,0.488163
sensor_431,0.487229
sensor_085,0.480197
sensor_434,0.452013


groupby() : 합격/불합격 그룹별로 특정 센서들의 평균값과 표준 편차를 한 번에 구한다.

### 📌 7장 결과 해석 — 그래서 무엇을 알게 됐나?

**왜 하나?** 1,567개 제품을 합격/불합격으로 묶어(`groupby`) 센서 평균을 비교하면, **불량품은 정상품과 무엇이 다른가**에 대한 단서를 얻을 수 있다. 차이가 큰 센서가 QualiGuard에서 우선 감시할 후보가 된다.

**① 처음 센서 3개 (sensor_000\~002)**
합격 3014.95 / 불합격 3007.53처럼 평균 차이가 표준편차(약 70\~90)에 비해 거의 없다. → 이 3개로는 불량을 구분할 수 없다.

**② 합격/불합격 차이가 큰 센서 상위 10개** (차이 ÷ 표준편차로 단위를 맞춰 `sort_values`)

| 순위 | 센서 | 차이(표준편차 배) | 결측 비율 | 판단 |
|---|---|---|---|---|
| 1 | sensor_059 | 0.62 | 0.4% | 후보 |
| 2 | sensor_103 | 0.61 | 0.1% | 후보 |
| 3 | sensor_510 | 0.53 | 0.1% | 후보 |
| 4 | sensor_348 | 0.52 | 1.5% | 후보 |
| 5 | sensor_158 | 0.52 | 91% | 신뢰 불가 |
| 6 | sensor_111 | 0.50 | 65% | 신뢰 불가 |
| 7 | sensor_293 | 0.49 | 91% | 신뢰 불가 |
| 8 | sensor_431 | 0.49 | 0.1% | 후보 |
| 9 | sensor_085 | 0.48 | 86% | 신뢰 불가 |
| 10 | sensor_434 | 0.45 | 0.1% | 후보 |

- 결측이 많은 센서는 남은 값 몇 개로 평균을 내서 차이가 우연히 커 보일 수 있다. → 믿을 만한 후보는 **6개 (059, 103, 510, 348, 431, 434)**. 8장에서 결측 많은 센서를 지우는 이유이기도 하다.
- 1위 sensor_059도 중앙값이 합격 0.72 / 불합격 5.52로 다르지만 값 범위가 크게 겹친다. → **센서 하나로는 불량을 못 잡고, 여러 센서를 함께 보는 머신러닝 모델이 필요하다.**

**③ 날짜별 불량** — 월별로 보면 7월 22.2%(63건 중 14건), 8월 9.2%(555건 중 51건), 9월 2.9%(590건 중 17건), 10월 6.1%(359건 중 22건). 불량이 **7\~8월에 몰려 있다.** 날짜로 보면 8월 10일은 12건 중 5건(41.7%)이 불합격 → 실제 공장이라면 그날 설비·원료를 조사하는 출발점이 된다.

> 정렬 기준에 따라 결론이 달라진다: **불합격 건수**로 정렬하면 10월 5일(32건 중 6건)이 1위, **불합격률**로 정렬하면 8월 10일이 1위. 무엇으로 `sort_values` 할지 의식해서 골라야 한다.

*   첫 번째 .sum(): 센서별 결측치 개수를 구함(열 단위 합계)
*   두 번째 .sum() : 센서별 개수들을 다시 총합함(전체 총합)



In [ ]:
# ======================================================
# 8. 결측치 다루기 - [L5] isnull, fillna( + drop)
# ======================================================
missing_ratio = df[sensor_cols].isnull().mean().sort_values(ascending=False)
print("전체 결측치 개수 : ", int(df[sensor_cols].isnull().sum().sum()))
print(missing_ratio.head(10).round(3))
print()
print()

drop_missing = missing_ratio[missing_ratio > 0.5].index
drop_const = n_unique[n_unique <= 1].index
drop_cols = drop_missing.union(drop_const)
print(f'결측 50% 초과 {len(drop_missing)}개, 값 1개뿐 {len(drop_const)}개 -> 삭제 {len(drop_cols)}개')
print()



clean = df.drop(columns=drop_cols)
keep_sensors = [ c for c in clean.columns if c.startswith('sensor_')]
clean[keep_sensors] = clean[keep_sensors].fillna(clean[keep_sensors].median())
print()
print()

print('남은 센서 : ', len(keep_sensors))
print('남은 결측치 : ', clean[keep_sensors].isnull().sum().sum()) # 0이면 성공

전체 결측치 개수 :  41951
sensor_292    0.912
sensor_293    0.912
sensor_158    0.912
sensor_157    0.912
sensor_492    0.856
sensor_085    0.856
sensor_358    0.856
sensor_220    0.856
sensor_244    0.650
sensor_517    0.650
dtype: float64


결측 50% 초과 28개, 값 1개뿐 116개 -> 삭제 144개



남은 센서 :  446
남은 결측치 :  0




1.   n_unique <= 1 : 앞서 구한 각 센서별 고유값 개수(n_unique) 중에서 값이 1개 이하인 것들만 True, 나머지는 False로 표시하는 조건식(마스크)이다.
2.   n_unique[..] : 그 조건식에서 맞는 센서들만 골라내어, 변화가 없는 문제의 센서들만 모아놓은 작은 시리즈(Series)를 만든다.
3. index : 판다스 시리즈나 데이터프레임에서 .index를 붙이면 그 데이터들의 행 이름만 리스트 형태로 싹 추출한다.




### 📌 8장 결과 해석 — 왜 결측치를 처리해야 하나?

**왜 하나?** scikit-learn 모델 대부분은 빈칸(NaN)이 하나라도 있으면 오류를 내고 멈춘다. Intermediate ML로 넘어가기 전에 반드시 처리해야 한다.

**SECOM의 결측 상태**
- 빈칸: 전체 924,530칸 중 **41,951칸 (약 4.5%)**
- 590개 센서 중 **538개에 빈칸이 하나 이상** 있다. → 빈칸 있는 행을 지우면(`dropna`) 데이터가 거의 다 사라지므로 이 데이터에는 쓸 수 없다.

**처리 결과**

| 처리 | 대상 | 이유 |
|---|---|---|
| 열 삭제 | 절반 넘게 빈 센서 28개 | 대부분을 지어낸 값으로 채워야 해서 오히려 왜곡된다 (7장의 sensor_158, 293 등) |
| 열 삭제 | 값이 하나뿐인 센서 116개 | 모든 제품에서 값이 같아 합격/불합격을 구분하는 정보가 0이다 |
| 중앙값으로 채우기 | 남은 446개 센서의 빈칸 | 평균보다 극단값에 덜 흔들린다 (sensor_059 최댓값 168이 평균을 끌어올리는 것처럼) |

→ 센서 **590개 → 446개**, 남은 빈칸 **0개**.

> 주의: 실제 모델 평가에서는 중앙값을 학습 데이터에서만 계산해야 한다 (Intermediate ML 'Pipelines'). 여기서는 Pandas 복습이라 전체 데이터로 계산했다.

In [ ]:
# ======================================================
# 9. 결과 저장하기 - [L1] to_csv
# ======================================================
cols_out = ['timestamp', 'label', 'is_fail'] + keep_sensors
clean[cols_out].to_csv(f'{DATA_DIR}/secom_clean.csv', index=False)
print()
print()


check = pd.read_csv(f'{DATA_DIR}/secom_clean.csv', parse_dates=['timestamp'])
print(check.shape)
check.head(3)



(1567, 449)


,timestamp,label,is_fail,sensor_000,sensor_001,sensor_002,sensor_003,sensor_004,sensor_006,sensor_007,...,sensor_576,sensor_577,sensor_582,sensor_583,sensor_584,sensor_585,sensor_586,sensor_587,sensor_588,sensor_589
0,2008-07-19 11:55:00,-1,0,3030.93,2564.00,2187.7333,1411.1265,1.3602,97.6133,0.1242,...,1.6765,14.9509,0.5005,0.0118,0.0035,2.3630,0.0205,0.0148,0.0046,71.9005
1,2008-07-19 12:32:00,-1,0,3095.78,2465.14,2230.4222,1463.6606,0.8294,102.3433,0.1247,...,1.1065,10.9003,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045
2,2008-07-19 13:17:00,1,1,2932.61,2559.94,2186.4111,1698.0172,1.5102,95.4878,0.1241,...,2.0952,9.2721,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602


### 📌 9장 결과 해석 — 그래서 결과가 어떻다는 건가?

Pandas 단계만으로 "어떤 제품이 불량이 될지"를 알 수는 없다. 그건 Intermediate ML의 몫이다. 오늘 Pandas 단계의 결과는 두 가지다.

**결과 1. 모델에 바로 넣을 수 있는 깨끗한 표 `secom_clean.csv`**
- 1,567행 × 449열 (시각·라벨·is_fail 3열 + 센서 446열)
- 빈칸 없음, 쓸모없는 센서 없음, 날짜는 날짜형
- 다음 복습에서 `pd.read_csv('secom_clean.csv')` 한 줄로 바로 모델 학습을 시작할 수 있다.

**결과 2. 데이터를 들여다보고 얻은 단서**
1. 불합격이 6.6%뿐인 **불균형 데이터** → 정확도가 아니라 불량을 얼마나 잡아냈는지(재현율 등)로 평가해야 한다.
2. 불량과 관련 있어 보이는 **센서 후보 6개** (059, 103, 510, 348, 431, 434) — 단, 하나만으로는 구분이 안 된다.
3. 불량이 **7\~8월에 집중** → 시간에 따라 공정 상태가 변했을 가능성이 있다.

**한 줄 정리:** 7장은 어디를 봐야 하는지 단서를 찾는 단계, 8장은 모델이 먹을 수 있게 데이터를 다듬는 단계, 9장은 그 결과물을 다음 단계로 넘기는 단계.

**다음 복습과의 연결**
- Data Cleaning: 중앙값 채우기를 다른 방식(평균, 앞 값으로 채우기 등)과 비교해 보기
- Intermediate ML: `secom_clean.csv`로 모델을 학습해서 후보 6개가 실제로 중요 변수로 뽑히는지 확인 → "Pandas로 세운 가설을 ML로 검증"